# 02 Silver: bronze to cleaned tables (incremental)

This notebook builds the `silver` schema from the `bronze` schema in `data/crossselliq.duckdb`. Run `01_bronze.ipynb` first.

Silver rules:

- Correct data types instead of text.
- Only the columns that are needed; bronze keeps everything, so a dropped column can always be added back.
- No transaction rows are removed: every bronze transaction row becomes exactly one silver row.
- The bronze metadata columns `_source_file` and `_loaded_at` are not carried over.
- Silver reads only from bronze, never from `data/raw`.

## How the incremental load works

- `silver.load_log` records every bronze file that silver has processed. On each run, only bronze files that are in `bronze.load_log` but not yet in `silver.load_log` are processed.
- Transactions are events that never change afterwards, so new rows are appended (`INSERT`).
- Customers and articles describe a current state that could change, so they are upserted with `MERGE`: new keys are inserted, existing keys are updated (SCD1, the latest snapshot wins). Files are processed in bronze load order.
- Processing a file and writing its log entry happen in one database transaction, so a file is never half processed or processed twice.
- If bronze was rebuilt (a file got a new bronze load time), silver can no longer trust its log, so it rebuilds itself from scratch.

Before running: close any other notebook or program that has the database open.

## Reading guide

Follow the notebook from top to bottom: connect to Bronze, define the cleaned tables, decide which files need processing, apply the cleaning rules, check the result, and inspect small summaries. Run all cells in order; the database connection closes in the final cell.

## 1. Settings and Bronze input

Keep `FULL_REFRESH = False` for normal incremental runs. The setup code checks that the DuckDB database and `bronze.load_log` exist before Silver reads any data.

In [1]:
# True rebuilds silver from scratch: all silver tables and the silver load log are dropped and every bronze file is processed again.
FULL_REFRESH = False

In [2]:
import time
from pathlib import Path

import duckdb

# Works whether the notebook runs from the project root or from notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "crossselliq.duckdb"

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}. Run 01_bronze.ipynb first.")

con = duckdb.connect(str(DB_PATH))


def table_exists(schema, table):
    return con.execute(
        "SELECT count(*) FROM information_schema.tables WHERE table_schema = $schema AND table_name = $table",
        {"schema": schema, "table": table},
    ).fetchone()[0] == 1


if not table_exists("bronze", "load_log"):
    con.close()
    raise RuntimeError("bronze.load_log not found. Run the incremental 01_bronze.ipynb first.")

print(f"Database: {DB_PATH}")

Database: c:\Users\diazv\Music\crosssell-iq\data\crossselliq.duckdb


## 2. Define the Silver tables and cleaning rules

The tables are created with explicit types. `customer_id` and `article_id` are primary keys, so the database itself refuses a duplicate customer or article.

### silver.customers

| Column | Rule |
|---|---|
| `customer_id` | unchanged; 64-character hexadecimal hash, unique |
| `fn`, `active` | `BOOLEAN`: `1.0` becomes `TRUE`, empty stays `NULL` |
| `club_member_status` | unchanged, empty stays `NULL` |
| `fashion_news_frequency` | `None` and `NONE` merged into `NONE` |
| `age` | `INTEGER` |
| `postal_code` | dropped |

Why empty `fn` and `active` stay `NULL` instead of `FALSE`: the dataset does not document what an empty value means, so `NULL` (unknown) is the honest choice.

Why `postal_code` is dropped: it is not needed for the baseline, and storing less personal data is safer (data minimisation). It is still available in bronze.

Limitation: customer attributes are a single snapshot without dates, probably the state at the end of the dataset (September 2020). There is no change history, so no SCD2. When these columns are used as model features in Milestone 4, they may contain information from after the reference date.

### silver.articles

Only the readable name columns are kept. The numeric code columns (`product_type_no`, `colour_group_code`, ...) hold the same information as the names, and `detail_desc` is a long free-text description that the project does not use.

`product_group_name` is the recommendation category. The value `Unknown` stays in silver, because silver describes the data as it is. Whether `Unknown` may be recommended is a business rule that belongs in gold.

### silver.transactions

| Column | Rule |
|---|---|
| `transaction_date` | `DATE`, from `t_dat` |
| `customer_id`, `article_id` | unchanged |
| `price` | `DOUBLE`; a scaled value from Kaggle, not an amount in euros |
| `sales_channel_id` | `TINYINT` (values 1 and 2) |

Identical rows are kept. The dataset has no quantity column: buying the same article twice on the same day gives two identical rows, so these are real purchases, not errors.

In [3]:
SILVER_TABLES_DDL = {
    "customers": """
        CREATE TABLE IF NOT EXISTS silver.customers (
            customer_id            VARCHAR PRIMARY KEY,
            fn                     BOOLEAN,
            active                 BOOLEAN,
            club_member_status     VARCHAR,
            fashion_news_frequency VARCHAR,
            age                    INTEGER
        )""",
    "articles": """
        CREATE TABLE IF NOT EXISTS silver.articles (
            article_id         VARCHAR PRIMARY KEY,
            product_code       VARCHAR,
            prod_name          VARCHAR,
            product_type_name  VARCHAR,
            product_group_name VARCHAR,
            colour_group_name  VARCHAR,
            department_name    VARCHAR,
            index_group_name   VARCHAR,
            section_name       VARCHAR,
            garment_group_name VARCHAR
        )""",
    "transactions": """
        CREATE TABLE IF NOT EXISTS silver.transactions (
            transaction_date DATE NOT NULL,
            customer_id      VARCHAR NOT NULL,
            article_id       VARCHAR NOT NULL,
            price            DOUBLE,
            sales_channel_id TINYINT
        )""",
}

LOAD_LOG_DDL = """
    CREATE TABLE IF NOT EXISTS silver.load_log (
        file_name        VARCHAR PRIMARY KEY,
        table_name       VARCHAR NOT NULL,
        bronze_loaded_at TIMESTAMP NOT NULL,
        row_count        BIGINT NOT NULL,
        processed_at     TIMESTAMP NOT NULL
    )"""

## 3. Decide which Bronze files to process

Silver is rebuilt from scratch when `FULL_REFRESH` is `True`, when `silver.load_log` does not exist yet, or when bronze was rebuilt after silver last processed a file. Otherwise only new bronze files are processed.

In [4]:
con.execute("CREATE SCHEMA IF NOT EXISTS silver")

rebuild_reason = None
if FULL_REFRESH:
    rebuild_reason = "FULL_REFRESH is True"
elif not table_exists("silver", "load_log"):
    rebuild_reason = "no silver load log found"
else:
    reloaded_in_bronze = con.execute(
        """
        SELECT count(*)
        FROM silver.load_log s
        LEFT JOIN bronze.load_log b USING (file_name)
        WHERE b.loaded_at IS DISTINCT FROM s.bronze_loaded_at
        """
    ).fetchone()[0]
    if reloaded_in_bronze:
        rebuild_reason = f"{reloaded_in_bronze} file(s) were reloaded or removed in bronze"

if rebuild_reason:
    print(f"Rebuilding silver from scratch ({rebuild_reason}).")
    # Table names come from the fixed SILVER_TABLES_DDL mapping, not from user input.
    for table_name in [*SILVER_TABLES_DDL, "load_log"]:
        con.execute(f"DROP TABLE IF EXISTS silver.{table_name}")

for ddl in [*SILVER_TABLES_DDL.values(), LOAD_LOG_DDL]:
    con.execute(ddl)

new_files = con.execute(
    """
    SELECT b.file_name, b.table_name, b.loaded_at
    FROM bronze.load_log b
    ANTI JOIN silver.load_log s USING (file_name)
    ORDER BY b.loaded_at, b.file_name
    """
).fetchall()

print(f"Bronze files already in silver: {con.execute('SELECT count(*) FROM silver.load_log').fetchone()[0]}")
print(f"New bronze files to process:    {len(new_files)}")

Rebuilding silver from scratch (no silver load log found).
Bronze files already in silver: 0
New bronze files to process:    3


## 4. Clean and process new Bronze files

The transformation queries are grouped below by table: `customers` keeps one current record per customer, `articles` keeps the product names and recommendation category, and `transactions` keeps every purchase while converting its date and numeric fields. The processing function then applies the right query to each new Bronze file and records it in `silver.load_log`.

Each query reads only the rows of one bronze file (`WHERE _source_file = $file_name`).

`MERGE ... UPDATE BY NAME` and `INSERT BY NAME` match columns by name. The shorter `UPDATE SET *` matches by position, which would silently put values in the wrong column if the column order ever differed.

The first run processes all 31.8 million transactions, so it takes a while. Later runs without new bronze files finish immediately.

In [5]:
CLEANED_CUSTOMERS_SQL = """
    SELECT
        customer_id,
        CASE WHEN FN = '1.0' THEN TRUE END AS fn,
        CASE WHEN Active = '1.0' THEN TRUE END AS active,
        club_member_status,
        CASE
            WHEN fashion_news_frequency = 'None' THEN 'NONE'
            ELSE fashion_news_frequency
        END AS fashion_news_frequency,
        CAST(age AS INTEGER) AS age
    FROM bronze.customers
    WHERE _source_file = $file_name
"""

CLEANED_ARTICLES_SQL = """
    SELECT
        article_id,
        product_code,
        prod_name,
        product_type_name,
        product_group_name,
        colour_group_name,
        department_name,
        index_group_name,
        section_name,
        garment_group_name
    FROM bronze.articles
    WHERE _source_file = $file_name
"""

CLEANED_TRANSACTIONS_SQL = """
    SELECT
        CAST(t_dat AS DATE) AS transaction_date,
        customer_id,
        article_id,
        CAST(price AS DOUBLE) AS price,
        CAST(sales_channel_id AS TINYINT) AS sales_channel_id
    FROM bronze.transactions
    WHERE _source_file = $file_name
"""


def merge_sql(table_name, key, cleaned_sql):
    return f"""
        MERGE INTO silver.{table_name} AS target
        USING ({cleaned_sql}) AS source
        ON target.{key} = source.{key}
        WHEN MATCHED THEN UPDATE BY NAME
        WHEN NOT MATCHED THEN INSERT BY NAME
    """


# Transactions are appended in date order, so DuckDB can skip data quickly when filtering on a period.
PROCESS_SQL = {
    "customers": merge_sql("customers", "customer_id", CLEANED_CUSTOMERS_SQL),
    "articles": merge_sql("articles", "article_id", CLEANED_ARTICLES_SQL),
    "transactions": f"INSERT INTO silver.transactions BY NAME {CLEANED_TRANSACTIONS_SQL} ORDER BY transaction_date",
}


def process_file(con, file_name, table_name, bronze_loaded_at):
    """Process one bronze file into silver and record it in the silver load log, atomically."""
    con.begin()
    try:
        row_count = con.execute(PROCESS_SQL[table_name], {"file_name": file_name}).fetchone()[0]
        con.execute(
            """
            INSERT INTO silver.load_log (file_name, table_name, bronze_loaded_at, row_count, processed_at)
            VALUES ($file_name, $table_name, $bronze_loaded_at, $row_count, current_timestamp AT TIME ZONE 'UTC')
            """,
            {
                "file_name": file_name,
                "table_name": table_name,
                "bronze_loaded_at": bronze_loaded_at,
                "row_count": row_count,
            },
        )
        con.commit()
    except Exception:
        con.rollback()
        raise
    return row_count


if not new_files:
    print("No new bronze files: silver is up to date.")

for file_name, table_name, bronze_loaded_at in new_files:
    start = time.perf_counter()
    row_count = process_file(con, file_name, table_name, bronze_loaded_at)
    method = "appended" if table_name == "transactions" else "merged"
    print(f"{file_name:<24} -> silver.{table_name:<13} {row_count:>12,} rows {method} in {time.perf_counter() - start:.1f} s")

articles.csv             -> silver.articles           105,542 rows merged in 0.6 s
customers.csv            -> silver.customers        1,371,980 rows merged in 6.4 s
transactions_train.csv   -> silver.transactions    31,788,324 rows appended in 6.5 s


## 5. Check the cleaned data

1. Every bronze file has been processed into silver.
2. Transactions: silver has exactly as many rows as the processed bronze files. Customers and articles: one silver row per unique key in bronze.
3. Expected columns and data types.
4. Keys are never empty; `customer_id` has the 64-character hex format.
5. Every transaction refers to an existing customer and article.
6. Cleaning rules: `fn` and `active` match the latest bronze snapshot, and only the expected values remain.

The last cell stops with an error if any check fails.

In [6]:
EXPECTED_SCHEMAS = {
    "customers": {
        "customer_id": "VARCHAR",
        "fn": "BOOLEAN",
        "active": "BOOLEAN",
        "club_member_status": "VARCHAR",
        "fashion_news_frequency": "VARCHAR",
        "age": "INTEGER",
    },
    "articles": {
        "article_id": "VARCHAR",
        "product_code": "VARCHAR",
        "prod_name": "VARCHAR",
        "product_type_name": "VARCHAR",
        "product_group_name": "VARCHAR",
        "colour_group_name": "VARCHAR",
        "department_name": "VARCHAR",
        "index_group_name": "VARCHAR",
        "section_name": "VARCHAR",
        "garment_group_name": "VARCHAR",
    },
    "transactions": {
        "transaction_date": "DATE",
        "customer_id": "VARCHAR",
        "article_id": "VARCHAR",
        "price": "DOUBLE",
        "sales_channel_id": "TINYINT",
    },
}
KEYS = {"customers": "customer_id", "articles": "article_id"}

results = []


def check(name, passed):
    results.append((name, passed))
    print(f"  {'OK  ' if passed else 'FAIL'} {name}")


def scalar(sql):
    return con.execute(sql).fetchone()[0]


print("load log")
check("every bronze file is processed into silver",
      scalar("SELECT count(*) FROM bronze.load_log ANTI JOIN silver.load_log USING (file_name)") == 0)

# Table and key names come from the fixed mappings above, not from user input.
for table_name, expected_schema in EXPECTED_SCHEMAS.items():
    silver_rows = scalar(f"SELECT count(*) FROM silver.{table_name}")
    if table_name == "transactions":
        expected_rows = scalar("SELECT count(*) FROM bronze.transactions")
        rule = "row count matches bronze"
    else:
        expected_rows = scalar(f"SELECT count(DISTINCT {KEYS[table_name]}) FROM bronze.{table_name}")
        rule = f"one row per unique {KEYS[table_name]} in bronze"
    actual_schema = {row[0]: row[1] for row in con.execute(f"DESCRIBE silver.{table_name}").fetchall()}

    print(f"silver.{table_name}: {silver_rows:,} rows")
    check(rule, silver_rows == expected_rows)
    check("columns and types as expected", actual_schema == expected_schema)

check("appended transaction rows match the silver load log",
      scalar("SELECT count(*) FROM silver.transactions")
      == scalar("SELECT coalesce(sum(row_count), 0) FROM silver.load_log WHERE table_name = 'transactions'"))

print("keys")
check("customer_id is a 64-character hex hash",
      scalar("SELECT count(*) FROM silver.customers WHERE NOT regexp_full_match(customer_id, '[0-9a-f]{64}')") == 0)
check("transaction keys and date never empty",
      scalar("""SELECT count(*) FROM silver.transactions
                WHERE transaction_date IS NULL OR customer_id IS NULL OR article_id IS NULL""") == 0)

print("relationships")
check("every transaction has a known customer",
      scalar("SELECT count(*) FROM silver.transactions ANTI JOIN silver.customers USING (customer_id)") == 0)
check("every transaction has a known article",
      scalar("SELECT count(*) FROM silver.transactions ANTI JOIN silver.articles USING (article_id)") == 0)

print("cleaning rules")
check("fn and active match the latest bronze snapshot",
      scalar("""
          WITH latest AS (
              SELECT customer_id, FN, Active
              FROM bronze.customers
              QUALIFY row_number() OVER (PARTITION BY customer_id ORDER BY _loaded_at DESC) = 1
          )
          SELECT count(*)
          FROM silver.customers s
          JOIN latest b USING (customer_id)
          WHERE coalesce(b.FN = '1.0', FALSE) <> (s.fn IS NOT NULL)
             OR coalesce(b.Active = '1.0', FALSE) <> (s.active IS NOT NULL)
      """) == 0)
check("fn and active only TRUE or NULL",
      scalar("SELECT count(*) FROM silver.customers WHERE fn = FALSE OR active = FALSE") == 0)
check("fashion_news_frequency only NONE, Regularly, Monthly or NULL",
      scalar("""SELECT count(*) FROM silver.customers
                WHERE fashion_news_frequency NOT IN ('NONE', 'Regularly', 'Monthly')""") == 0)
check("sales_channel_id only 1 or 2",
      scalar("SELECT count(*) FROM silver.transactions WHERE sales_channel_id NOT IN (1, 2)") == 0)
check("price always positive",
      scalar("SELECT count(*) FROM silver.transactions WHERE price <= 0") == 0)

load log
  OK   every bronze file is processed into silver
silver.customers: 1,371,980 rows
  OK   one row per unique customer_id in bronze
  OK   columns and types as expected
silver.articles: 105,542 rows
  OK   one row per unique article_id in bronze
  OK   columns and types as expected
silver.transactions: 31,788,324 rows
  OK   row count matches bronze
  OK   columns and types as expected
  OK   appended transaction rows match the silver load log
keys
  OK   customer_id is a 64-character hex hash
  OK   transaction keys and date never empty
relationships
  OK   every transaction has a known customer
  OK   every transaction has a known article
cleaning rules
  OK   fn and active match the latest bronze snapshot
  OK   fn and active only TRUE or NULL
  OK   fashion_news_frequency only NONE, Regularly, Monthly or NULL
  OK   sales_channel_id only 1 or 2
  OK   price always positive


In [7]:
failed = [name for name, passed in results if not passed]
if failed:
    raise AssertionError(f"{len(failed)} silver check(s) failed: {failed}")
print(f"All {len(results)} silver checks passed.")

All 17 silver checks passed.


## 6. Inspect the result

The load log shows which Bronze files were processed. The small summaries show the transaction period, customers without purchases, and article counts per product category. These are checks and previews, not Gold tables.

In [8]:
con.sql("SELECT file_name, table_name, row_count, processed_at FROM silver.load_log ORDER BY processed_at, file_name").show()

print("Transaction period and size")
con.sql("""
    SELECT
        min(transaction_date) AS first_date,
        max(transaction_date) AS last_date,
        count(*) AS transactions,
        count(DISTINCT customer_id) AS buying_customers
    FROM silver.transactions
""").show()

print("Customers without any purchase (needed for API error handling)")
con.sql("""
    SELECT count(*) AS customers_without_purchases
    FROM silver.customers
    ANTI JOIN silver.transactions USING (customer_id)
""").show()

print("Articles per product_group_name (the recommendation category)")
con.sql("""
    SELECT product_group_name, count(*) AS articles
    FROM silver.articles
    GROUP BY product_group_name
    ORDER BY articles DESC
""").show(max_rows=25)

┌────────────────────────┬──────────────┬───────────┬────────────────────────────┐
│       file_name        │  table_name  │ row_count │        processed_at        │
│        varchar         │   varchar    │   int64   │         timestamp          │
├────────────────────────┼──────────────┼───────────┼────────────────────────────┤
│ articles.csv           │ articles     │    105542 │ 2026-10-02 09:39:59.592859 │
│ customers.csv          │ customers    │   1371980 │ 2026-10-02 09:40:00.171484 │
│ transactions_train.csv │ transactions │  31788324 │ 2026-10-02 09:40:06.554381 │
└────────────────────────┴──────────────┴───────────┴────────────────────────────┘

Transaction period and size
┌────────────┬────────────┬──────────────┬──────────────────┐
│ first_date │ last_date  │ transactions │ buying_customers │
│    date    │    date    │    int64     │      int64       │
├────────────┼────────────┼──────────────┼──────────────────┤
│ 2018-09-20 │ 2020-09-22 │     31788324 │          1362281

## 7. Close the database

Always close the connection, otherwise the database file stays locked for other programs.

In [9]:
con.close()